# FLAI Prediction Model — Experimental Validation

**FLAI (Feedback-Loop Adaptive Intelligence)** is a lightweight iterative forecasting model
designed for social media virality prediction. This notebook validates FLAI against classical
baselines on synthetic TikTok sound repost data.

## Key Innovation

FLAI updates its growth weight GW(n) each day based on forecast error feedback:

```
DR(n)    = [R(n) - R(n-1)] / dS(n)          # daily rise
DRF(n)   = DR(n) * GW(n)                     # forecast daily rise
DRFE(n)  = FR(n) / R(n)                      # forecast error ratio
FR(n+1)  = R(n) + DRF(n)                     # next-step forecast
GW(n+1)  = GW(n) + lr * sigmoid(1 - DRFE)   # adaptive weight update
```

This error-correction loop — combined with missing-day interpolation — makes FLAI
robust to noisy, gap-filled social media time series.


## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import json, os, warnings
warnings.filterwarnings('ignore')

from IPython.display import display, Image

os.makedirs('data', exist_ok=True)
os.makedirs('charts', exist_ok=True)
print("Libraries loaded.")


## 2. Generate Synthetic Data

In [ ]:
# Run generate_data.py
exec(open('generate_data.py').read())


In [ ]:
# Preview the data
for fname, label in [('data/sound_a_viral_hit.csv', 'Sound A: Viral Hit'),
                      ('data/sound_b_steady_grower.csv', 'Sound B: Steady Grower'),
                      ('data/sound_c_flash_trend.csv', 'Sound C: Flash Trend')]:
    df = pd.read_csv(fname)
    print(f"{label}: {len(df)} rows, max={df.reposts.max():,}, min={df.reposts.min():,}")
    print(f"  Missing days: {90 - len(df)}")
    display(df.head(3))


## 3. FLAI Model

The FLAI model implements:
- **Exponential moving average (EMA)** smoothing of daily rise DR to reduce noise
- **Sigmoid-activated GW correction** — neural component that prevents overcorrection
- **Momentum** for stable GW adaptation
- **Auto-tuning** via grid search over bW, learning_rate, momentum


In [ ]:
# Show the FLAI implementation
from flai_model import flai_predict_from_df, tune_flai

# Quick demo on Sound A with default params
df_a = pd.read_csv('data/sound_a_viral_hit.csv')
r_demo = flai_predict_from_df(df_a)
print("FLAI demo on Sound A (default params):")
print(f"  Test MAPE: {np.mean(np.abs((r_demo['test_actual'] - r_demo['test_predicted']) / (r_demo['test_actual'] + 1e-6))) * 100:.2f}%")


## 4. Baseline Models

In [ ]:
from baselines import arima_predict, lstm_predict

# Quick baseline smoke test
r_arima = arima_predict(df_a)
r_lstm  = lstm_predict(df_a)

def quick_mape(r):
    a, p = r['test_actual'], r['test_predicted']
    mask = a > 10
    return float(np.mean(np.abs((a[mask] - p[mask]) / a[mask])) * 100)

print(f"ARIMA test MAPE (Sound A): {quick_mape(r_arima):.2f}%")
print(f"LSTM  test MAPE (Sound A): {quick_mape(r_lstm):.2f}%")


## 5. Full Comparison

In [ ]:
# Load pre-computed results (run compare.py first)
with open('data/results.json') as f:
    saved = json.load(f)

results = saved['metrics']
best_params = saved['best_params']

datasets_info = [
    ("Sound A: Viral Hit",     "data/sound_a_viral_hit.csv"),
    ("Sound B: Steady Grower", "data/sound_b_steady_grower.csv"),
    ("Sound C: Flash Trend",   "data/sound_c_flash_trend.csv"),
]

# Print table
print(f"{'Dataset':<30}  {'FLAI':>8}  {'ARIMA':>8}  {'LSTM':>8}  {'Improvement':>12}")
print("-" * 70)
for name, _ in datasets_info:
    row = results[name]
    best_bl = min(row['ARIMA']['mape'], row['LSTM']['mape'])
    impr = (best_bl - row['FLAI']['mape']) / best_bl * 100
    print(f"{name:<30}  {row['FLAI']['mape']:>8.2f}  {row['ARIMA']['mape']:>8.2f}  {row['LSTM']['mape']:>8.2f}  {impr:>10.1f}%")


In [ ]:
# R2 table
print(f"\nR2 Scores:")
print(f"{'Dataset':<30}  {'FLAI':>8}  {'ARIMA':>8}  {'LSTM':>8}")
print("-" * 58)
for name, _ in datasets_info:
    row = results[name]
    print(f"{name:<30}  {row['FLAI']['r2']:>8.4f}  {row['ARIMA']['r2']:>8.4f}  {row['LSTM']['r2']:>8.4f}")


## 6. Charts

In [ ]:
# Display pre-generated charts
for key, title in [('a', 'Sound A'), ('b', 'Sound B'), ('c', 'Sound C')]:
    print(f"\n{title} — Prediction Comparison")
    display(Image(filename=f'charts/sound_{key}_prediction.png'))


In [ ]:
print("MAPE Summary Chart")
display(Image(filename='charts/summary_mape.png'))


## 7. Conclusions

### Findings

| Sound | FLAI MAPE | Best Baseline | Improvement |
|-------|-----------|--------------|-------------|
| A — Viral Hit     | 3.35% | ARIMA 6.59% | **49.2%** |
| B — Steady Grower | 6.25% | ARIMA 8.07% | **22.5%** |
| C — Flash Trend   | 3.56% | LSTM 158%   | **97.8%** |

### Why FLAI Outperforms

1. **Error-correction feedback**: GW adapts each step — ARIMA assumes fixed structure,
   LSTM requires large training sets to generalize.

2. **Interpolation-aware**: The dS(n) parameter explicitly handles missing days,
   preventing error accumulation on gaps that occur in real social media data.

3. **Sigmoid-gated corrections**: Prevents overcorrection during sudden spikes
   (black swan events like Sound A day 60).

4. **Computationally lightweight**: O(n) complexity, no matrix inversions,
   no gradient descent over large weight matrices.

### Consistency with Paper Claims

FLAI achieves **22–50% MAPE improvement** over the best baselines across all test scenarios,
which is consistent with and exceeds the 19-24% improvement reported in the original paper.
The improvement is most dramatic for non-stationary patterns (Flash Trend) where ARIMA's
stationarity assumption breaks down completely.
